# 01 · Data Quality Assessment & Cleaning

**Goal:** turn three raw exports (sessions, clickstream events, orders) into trustworthy,
analysis-ready tables, and document every decision so the numbers can be audited.

**Why this matters:** in funnel analysis, small data problems create big wrong conclusions.
Bot traffic inflates sessions and *lowers* conversion; double-fired pixels inflate add-to-cart;
QA test orders inflate order counts and drag down AOV. We fix each one before any analysis.

| Step | What we check |
|---|---|
| 1 | Profile raw tables: shape, types, nulls, duplicates |
| 2 | Standardise labels and parse timestamps |
| 3 | Detect and remove bot traffic (behavioural rule) |
| 4 | Remove test & duplicate orders |
| 5 | Build a session-level funnel table and validate funnel logic |

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import clean  # our pipeline module

pd.set_option("display.max_colwidth", 80)
s_raw, ev_raw, o_raw = clean.load_raw()

## 1 · Profile the raw data

In [2]:
def profile(df, name):
    p = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (100 * df.isna().mean()).round(2),
        "unique": df.nunique(),
        "example": df.iloc[0],
    })
    print(f"{name}: {len(df):,} rows x {df.shape[1]} cols | exact duplicate rows: {df.duplicated().sum():,}")
    return p

profile(s_raw, "sessions")

sessions: 82,000 rows x 10 cols | exact duplicate rows: 0


,dtype,nulls,null_%,unique,example
session_id,str,0,0.00,82000,S1000000
user_id,str,0,0.00,32345,U143999
session_start,str,0,0.00,79445,2026-08-01 00:07:34
device,str,945,1.15,3,mobile
channel,str,0,0.00,14,organic_search
city_tier,str,0,0.00,3,tier_2
landing_page,str,0,0.00,4,category
is_returning,int64,0,0.00,2,0
page_load_ms,int64,0,0.00,5955,3546
ab_group,str,0,0.00,3,not_in_test


In [3]:
profile(ev_raw, "events")

events: 285,621 rows x 6 cols | exact duplicate rows: 2,195


,dtype,nulls,null_%,unique,example
session_id,str,0,0.00,82000,S1018370
user_id,str,0,0.00,32345,U130319
event_name,str,0,0.00,6,product_view
event_ts,str,0,0.00,274336,2026-08-23 10:56:02
product_category,str,82654,28.94,5,fashion
payment_method,str,279717,97.93,4,NaN


In [4]:
profile(o_raw, "orders")

orders: 2,789 rows x 9 cols | exact duplicate rows: 20


,dtype,nulls,null_%,unique,example
order_id,str,0,0.0,2769,O500000
session_id,str,0,0.0,2734,S1000100
user_id,str,0,0.0,2393,U142988
order_ts,str,0,0.0,2733,2026-08-01 08:15:01
product_category,str,0,0.0,5,fashion
payment_method,str,0,0.0,4,upi
gross_value_inr,float64,0,0.0,2729,1823.98
discount_inr,float64,0,0.0,1736,182.4
net_value_inr,float64,0,0.0,2721,1641.58


### Issues spotted while profiling
1. **`channel` has 14 distinct values for 7 real channels** — two tracking tools write labels differently.
2. **`device` is null for ~1.2% of sessions** — usually ad-blockers stopping the tracking script.
3. **`event_ts` is stored as text in two formats** (ISO and `DD/MM/YYYY`) — sorting as text would be wrong.
4. **Exact duplicate events** — a pixel firing twice.
5. **Orders with value ₹1 and IDs starting `TEST`** — QA test orders, plus a few duplicated order rows.
6. A suspicious cluster of sessions with `page_load_ms = 0` — checked below.

In [5]:
s_raw.channel.value_counts()

channel
organic_search    20341
paid_social       16615
paid_search       13642
direct            13148
email              5834
affiliate          3743
referral           3718
Organic Search     1350
Paid Social        1072
Paid Search         900
Direct              750
Email               375
Referral            267
Affiliate           245
Name: count, dtype: int64

In [6]:
ev_raw.event_ts.sample(8, random_state=2).to_frame()

,event_ts
52932,2026-08-02 22:09:41
11691,2026-10-28 22:22:22
151678,29/09/2026 13:51:27
117705,2026-08-28 15:09:51
176210,2026-09-17 21:06:59
194130,2026-10-16 23:51:19
159450,2026-10-03 14:12:41
282164,2026-08-04 17:56:20


## 2 · Bot traffic — finding it by behaviour
A human doesn't view 20+ products in under a minute. We flag a session as a bot if it has
**≥ 15 product views within 60 seconds**, or a **page load time of 0 ms** (no real browser rendered the page).
We deliberately do *not* use the `user_id` text, because in real data bots don't label themselves.

In [7]:
ev_tmp = clean.clean_events(ev_raw, clean.Log())
agg = ev_tmp.groupby("session_id").agg(
    views=("event_name", lambda x: (x == "product_view").sum()),
    duration_s=("event_ts", lambda t: (t.max() - t.min()).total_seconds()))
agg["suspected_bot"] = (agg.views >= clean.BOT_MIN_VIEWS) & (agg.duration_s <= clean.BOT_MAX_DURATION_S)
agg.groupby("suspected_bot")[["views", "duration_s"]].describe().T.round(1)

suspected_bot       False   True 
views      count  80000.0  2000.0
           mean       1.7    24.0
           std        1.5     0.0
           min        0.0    24.0
           25%        0.0    24.0
           50%        2.0    24.0
           75%        3.0    24.0
           max       10.0    24.0
duration_s count  80000.0  2000.0
           mean     101.1    24.0
           std      114.9     0.0
           min        0.0    24.0
           25%        0.0    24.0
           50%       68.0    24.0
           75%      182.0    24.0
           max     1901.0    24.0

The two groups don't overlap at all: every suspected bot has 24 views in 24 seconds;
humans average fewer than 2 views and stay about 1.5 minutes. The rule is safe.

## 3 · Run the full cleaning pipeline
All rules live in `src/clean.py` so the notebook and the production script give identical results.

In [8]:
log, summary, funnel = clean.run()
log

,table,check,issue,action,rows_affected
0,sessions,channel labels,14 raw labels for 7 channels (e.g. 'Paid Social' vs 'paid_social'),lower-case + underscores,4959
1,sessions,missing device,device not captured (tracking script blocked),set to 'unknown' (kept — rows are still valid sessions),945
2,sessions,duplicate session_id,primary key must be unique,drop duplicates,0
3,events,timestamp format,two string formats in the export (ISO and DD/MM/YYYY),parse both formats explicitly,11288
4,events,unparseable timestamps,timestamp could not be parsed,drop,0
5,events,exact duplicate events,tracking pixel fired twice,drop exact duplicates,2386
6,orders,duplicate order rows,same order exported twice,drop exact duplicates,20
7,orders,QA test orders,"internal test orders (order_id 'TEST*', value Rs 1)",drop,35
8,orders,negative order value,refunds mixed into orders,drop (none expected),0
9,sessions,bot / crawler traffic,">= 15 product views within 60s, or page_load_ms = 0",exclude from all analysis,2000


In [9]:
summary

,table,raw_rows,clean_rows,removed,removed_pct
0,sessions,82000,80000,2000,2.44
1,events,285621,233235,52386,18.34
2,orders,2789,2734,55,1.97


## 4 · Impact of cleaning on the headline metric
What would we have reported if we had skipped cleaning?

In [10]:
raw_sessions = len(s_raw)
raw_orders = len(o_raw)
clean_sessions = len(funnel)
clean_orders = int(funnel.purchase.sum())
pd.DataFrame({
    "metric": ["sessions", "orders", "conversion rate %"],
    "raw (uncleaned)": [raw_sessions, raw_orders, round(100 * raw_orders / raw_sessions, 2)],
    "clean": [clean_sessions, clean_orders, round(100 * clean_orders / clean_sessions, 2)],
})

,metric,raw (uncleaned),clean
0,sessions,82000.0,80000.00
1,orders,2789.0,2734.00
2,conversion rate %,3.4,3.42


The difference looks small at the top line, but bots are concentrated on **desktop / direct /
electronics**, so without cleaning, those segments would look much worse than they are and
we'd send the product team after the wrong problem.

## 5 · The analysis-ready table: `session_funnel`
One row per human session, with a 0/1 flag for each funnel stage. This is the table every later
notebook and SQL query uses.

In [11]:
funnel.head()

,session_id,user_id,session_start,device,channel,city_tier,landing_page,is_returning,page_load_ms,ab_group,reached_session,product_view,add_to_cart,checkout_start,payment_attempt,purchase,n_product_views,revenue_inr
0,S1000000,U143999,2026-08-01 00:07:34,mobile,organic_search,tier_2,category,0,3546,not_in_test,1,0,0,0,0,0,0,0.0
1,S1000001,U149586,2026-08-01 00:11:40,mobile,paid_search,tier_2,category,0,2335,not_in_test,1,1,1,0,0,0,4,0.0
2,S1000002,U146756,2026-08-01 00:13:13,tablet,direct,tier_2,home,0,2511,not_in_test,1,0,0,0,0,0,0,0.0
3,S1000003,U148659,2026-08-01 00:14:52,mobile,direct,tier_2,category,0,3274,not_in_test,1,1,1,1,0,0,4,0.0
4,S1000004,U108714,2026-08-01 00:16:01,desktop,organic_search,tier_1,offer,0,2649,not_in_test,1,1,1,0,0,0,2,0.0


In [12]:
funnel[["reached_session", "product_view", "add_to_cart", "checkout_start",
        "payment_attempt", "purchase"]].sum().to_frame("sessions_reaching_stage")

,sessions_reaching_stage
reached_session,80000
product_view,56002
add_to_cart,8523
checkout_start,4676
payment_attempt,3120
purchase,2734


### ✅ Output
* `data/processed/sessions_clean.parquet`, `events_clean.parquet`, `orders_clean.parquet`, `session_funnel.parquet`
* `reports/data_quality_log.csv` — an auditable record of every rule and how many rows it touched
* All funnel-logic validations passed (no session skips a stage; every purchase has exactly one order)